# Module 06: Side-Channel Analysis Theory — Lab

This lab provides interactive exploration of power analysis concepts.

**Objectives:**
1. Compute Hamming Weight and Hamming Distance for arbitrary values
2. Simulate SPA by observing instruction-level power patterns
3. Perform a CPA attack on simulated AES S-box leakage
4. Run TVLA (t-test) to detect leakage in simulated traces

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

# Hamming Weight and Hamming Distance
print("=" * 60)
print("LEAKAGE MODELS: HAMMING WEIGHT & HAMMING DISTANCE")
print("=" * 60)

def hamming_weight(x):
    """Compute Hamming weight of an integer"""
    return bin(x).count('1')

def hamming_distance(x, y):
    """Compute Hamming distance between two integers"""
    return hamming_weight(x ^ y)

# AES S-box
AES_SBOX = [
    0x63,0x7C,0x77,0x7B,0xF2,0x6B,0x6F,0xC5,0x30,0x01,0x67,0x2B,0xFE,0xD7,0xAB,0x76,
    0xCA,0x82,0xC9,0x7D,0xFA,0x59,0x47,0xF0,0xAD,0xD4,0xA2,0xAF,0x9C,0xA4,0x72,0xC0,
    0xB7,0xFD,0x93,0x26,0x36,0x3F,0xF7,0xCC,0x34,0xA5,0xE5,0xF1,0x71,0xD8,0x31,0x15,
    0x04,0xC7,0x23,0xC3,0x18,0x96,0x05,0x9A,0x07,0x12,0x80,0xE2,0xEB,0x27,0xB2,0x75,
    0x09,0x83,0x2C,0x1A,0x1B,0x6E,0x5A,0xA0,0x52,0x3B,0xD6,0xB3,0x29,0xE3,0x2F,0x84,
    0x53,0xD1,0x00,0xED,0x20,0xFC,0xB1,0x5B,0x6A,0xCB,0xBE,0x39,0x4A,0x4C,0x58,0xCF,
    0xD0,0xEF,0xAA,0xFB,0x43,0x4D,0x33,0x85,0x45,0xF9,0x02,0x7F,0x50,0x3C,0x9F,0xA8,
    0x51,0xA3,0x40,0x8F,0x92,0x9D,0x38,0xF5,0xBC,0xB6,0xDA,0x21,0x10,0xFF,0xF3,0xD2,
    0xCD,0x0C,0x13,0xEC,0x5F,0x97,0x44,0x17,0xC4,0xA7,0x7E,0x3D,0x64,0x5D,0x19,0x73,
    0x60,0x81,0x4F,0xDC,0x22,0x2A,0x90,0x88,0x46,0xEE,0xB8,0x14,0xDE,0x5E,0x0B,0xDB,
    0xE0,0x32,0x3A,0x0A,0x49,0x06,0x24,0x5C,0xC2,0xD3,0xAC,0x62,0x91,0x95,0xE4,0x79,
    0xE7,0xC8,0x37,0x6D,0x8D,0xD5,0x4E,0xA9,0x6C,0x56,0xF4,0xEA,0x65,0x7A,0xAE,0x08,
    0xBA,0x78,0x25,0x2E,0x1C,0xA6,0xB4,0xC6,0xE8,0xDD,0x74,0x1F,0x4B,0xBD,0x8B,0x8A,
    0x70,0x3E,0xB5,0x66,0x48,0x03,0xF6,0x0E,0x61,0x35,0x57,0xB9,0x86,0xC1,0x1D,0x9E,
    0xE1,0xF8,0x98,0x11,0x69,0xD9,0x8E,0x94,0x9B,0x1E,0x87,0xE9,0xCE,0x55,0x28,0xDF,
    0x8C,0xA1,0x89,0x0D,0xBF,0xE6,0x42,0x68,0x41,0x99,0x2D,0x0F,0xB0,0x54,0xBB,0x16,
]

# Show HW distribution of S-box outputs
hw_dist = [0] * 9
for val in AES_SBOX:
    hw_dist[hamming_weight(val)] += 1

print("AES S-box output Hamming Weight distribution:")
for i, count in enumerate(hw_dist):
    bar = '#' * (count * 2)
    print(f"  HW={i}: {count:3d}  {bar}")

# Example: SPA on RSA square-and-multiply
print("\n" + "=" * 60)
print("SPA ON RSA SQUARE-AND-MULTIPLY")
print("=" * 60)

# Private key bits (small example)
d = 0b10110011  # = 179
bits = bin(d)[2:]
print(f"Private key d = {d} (binary: {bits})")

# Simulate power trace for square-and-multiply
np.random.seed(42)
samples_per_op = 20
trace = []
ops = []

for bit in bits:
    # Square operation
    sq_pattern = np.random.normal(1.0, 0.1, samples_per_op)
    trace.extend(sq_pattern)
    ops.append('S')
    
    if bit == '1':
        # Multiply operation (higher power)
        mul_pattern = np.random.normal(2.0, 0.1, samples_per_op)
        trace.extend(mul_pattern)
        ops.append('M')

trace = np.array(trace)

plt.figure(figsize=(14, 4))
plt.plot(trace, linewidth=0.5)
plt.title(f'SPA Trace for RSA (key bits: {bits})')
plt.xlabel('Sample')
plt.ylabel('Power')
plt.axhline(y=1.5, color='r', linestyle='--', alpha=0.5, label='Decision threshold')
plt.legend()
plt.tight_layout()
plt.show()

# Recover key from trace
n_ops = len(trace) // samples_per_op
recovered_bits = []
for i in range(n_ops):
    segment = trace[i*samples_per_op:(i+1)*samples_per_op]
    avg_power = np.mean(segment)
    if avg_power > 1.5:  # Multiply detected
        recovered_bits.append('1')
    else:  # Square only
        recovered_bits.append('0')
        if i < n_ops - 1:  # Skip next multiply (it's part of same cycle)
            pass

print(f"Recovered bits: {''.join(recovered_bits)}")
print(f"Original bits:  {bits}")
print(f"Note: SPA is vulnerable to countermeasures like constant-time implementations")

In [ ]:
# CPA Attack on AES S-box Leakage
print("=" * 60)
print("CPA ATTACK ON AES S-BOX LEAKAGE")
print("=" * 60)

np.random.seed(123)

# Parameters
n_traces = 500
n_samples = 100  # Time samples per trace
secret_key = [0x2B, 0x7E, 0x15, 0x16, 0x28, 0xAE, 0xD2, 0xA6,
              0xAB, 0xF7, 0x15, 0x88, 0x09, 0xCF, 0x4F, 0x3C]

# Generate simulated power traces
# Leakage: HW(S-box output) at a specific time sample
poa = 50  # Point of Attack (where SubBytes executes)

plaintexts = np.random.randint(0, 256, (n_traces, 16), dtype=np.uint8)
traces = np.random.normal(0, 0.5, (n_traces, n_samples))  # Noise floor

# Add leakage for all 16 key bytes at different time samples
for byte_idx in range(16):
    t_offset = 50 + byte_idx * 2  # Slightly offset per byte
    for i in range(n_traces):
        sbox_out = AES_SBOX[plaintexts[i, byte_idx] ^ secret_key[byte_idx]]
        hw = hamming_weight(sbox_out)
        traces[i, t_offset] += hw * 0.3 + np.random.normal(0, 0.1)

print(f"Traces shape: {traces.shape}")
print(f"Secret key: {['0x{:02X}'.format(k) for k in secret_key]}")

# Plot a few traces
plt.figure(figsize=(14, 4))
for i in range(5):
    plt.plot(traces[i], alpha=0.5, linewidth=0.5)
plt.title('Simulated Power Traces (first 5)')
plt.xlabel('Sample')
plt.ylabel('Power')
plt.tight_layout()
plt.show()

In [ ]:
# Perform CPA attack on byte 0 of the key
print("=" * 60)
print("CPA ATTACK: RECOVERING KEY BYTE 0")
print("=" * 60)

def pearson_correlation(x, y):
    """Compute Pearson correlation coefficient"""
    n = len(x)
    if n == 0:
        return 0
    mean_x = np.mean(x)
    mean_y = np.mean(y)
    dx = x - mean_x
    dy = y - mean_y
    numerator = np.sum(dx * dy)
    denominator = np.sqrt(np.sum(dx**2) * np.sum(dy**2))
    if denominator == 0:
        return 0
    return numerator / denominator

# Attack byte 0
byte_idx = 0
correlations = np.zeros((256, n_samples))

for k_guess in range(256):
    # Compute intermediate values for all traces
    intermediates = np.array([AES_SBOX[plaintexts[i, byte_idx] ^ k_guess] for i in range(n_traces)])
    intermediates = np.array([hamming_weight(v) for v in intermediates], dtype=float)
    
    # Correlate with each time sample
    for t in range(n_samples):
        correlations[k_guess, t] = pearson_correlation(intermediates, traces[:, t])

# Find best key guess
max_corr = np.max(np.abs(correlations), axis=1)
best_key = np.argmax(max_corr)

print(f"\nTrue key byte 0: 0x{secret_key[byte_idx]:02X}")
print(f"Recovered key byte: 0x{best_key:02X}")
print(f"Max correlation: {max_corr[best_key]:.4f}")
print(f"Rank of correct key: {np.argsort(-max_corr).tolist().index(secret_key[byte_idx]) + 1}")

# Plot correlation traces for top 5 key guesses
top5 = np.argsort(-max_corr)[:5]
plt.figure(figsize=(14, 5))
for k in top5:
    label = f"0x{k:02X} {'(TRUE)' if k == secret_key[byte_idx] else ''}"
    plt.plot(correlations[k], label=label, alpha=0.8)
plt.title(f'CPA Correlation Traces - Top 5 Key Guesses for Byte {byte_idx}')
plt.xlabel('Time Sample')
plt.ylabel('Correlation')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# TVLA (Test Vector Leakage Assessment)
print("=" * 60)
print("TVLA: TEST VECTOR LEAKAGE ASSESSMENT")
print("=" * 60)

# Generate fixed and random trace sets
n_tvla = 1000

# Fixed set: all plaintexts = 0x00
fixed_pt = np.zeros((n_tvla, 16), dtype=np.uint8)
fixed_traces = np.random.normal(0, 0.5, (n_tvla, n_samples))
for i in range(n_tvla):
    sbox_out = AES_SBOX[fixed_pt[i, 0] ^ secret_key[0]]
    fixed_traces[i, 50] += hamming_weight(sbox_out) * 0.3

# Random set: random plaintexts
random_pt = np.random.randint(0, 256, (n_tvla, 16), dtype=np.uint8)
random_traces = np.random.normal(0, 0.5, (n_tvla, n_samples))
for i in range(n_tvla):
    sbox_out = AES_SBOX[random_pt[i, 0] ^ secret_key[0]]
    random_traces[i, 50] += hamming_weight(sbox_out) * 0.3

# Compute Welch's t-test at each time sample
t_values = np.zeros(n_samples)
for t in range(n_samples):
    mean_fixed = np.mean(fixed_traces[:, t])
    mean_random = np.mean(random_traces[:, t])
    var_fixed = np.var(fixed_traces[:, t], ddof=1)
    var_random = np.var(random_traces[:, t], ddof=1)
    
    se = np.sqrt(var_fixed/n_tvla + var_random/n_tvla)
    if se > 0:
        t_values[t] = (mean_fixed - mean_random) / se

# Decision: |t| > 4.5 indicates leakage
threshold = 4.5
leaking_samples = np.where(np.abs(t_values) > threshold)[0]

print(f"Number of traces per set: {n_tvla}")
print(f"Leakage threshold: |t| > {threshold}")
print(f"Leaking time samples: {len(leaking_samples)}")
if len(leaking_samples) > 0:
    print(f"Leakage DETECTED at samples: {leaking_samples[:10]}{'...' if len(leaking_samples) > 10 else ''}")
else:
    print("No leakage detected")

# Plot t-test results
plt.figure(figsize=(14, 5))
plt.plot(t_values, linewidth=0.5)
plt.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold (±{threshold})')
plt.axhline(y=-threshold, color='r', linestyle='--')
plt.fill_between(range(n_samples), threshold, np.max(np.abs(t_values))+1,
                 where=np.abs(t_values) > threshold, color='red', alpha=0.3, label='Leakage')
plt.title('TVLA t-test Results')
plt.xlabel('Time Sample')
plt.ylabel('|t|')
plt.legend()
plt.tight_layout()
plt.show()

print("\nKey Insight: TVLA can detect leakage WITHOUT knowing the secret key.")
print("It is used for certification testing (FIPS 140-3) and before deploying countermeasures.")